# CC2017 – Deep Learning · Proyecto Final
## 01 · Exploración del dataset GTSRB

El German Traffic Sign Recognition Benchmark (GTSRB) contiene fotos de señales de tránsito de 43 clases tomadas desde un vehículo en movimiento. En este notebook se descarga el dataset, se revisa la distribución de clases y tamaños, y se define la partición train/val que usan los demás notebooks.

## Preparación · Librerías, reproducibilidad y dispositivo

In [ ]:
import sys, random, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
OUT = ROOT / 'outputs'; OUT.mkdir(exist_ok=True)

# FAST_DEV_RUN=True recorre pocos lotes por época para comprobar que todo corre; sus checkpoints llevan prefijo dev_.
FAST_DEV_RUN = False
MAX_BATCHES = 5 if FAST_DEV_RUN else None
PREFIX = 'dev_' if FAST_DEV_RUN else ''
NUM_WORKERS = 2
print('Dispositivo:', device, '| FAST_DEV_RUN:', FAST_DEV_RUN)

## 1. Descarga y partición
`kagglehub` descarga el dataset a su caché local, por lo que no se copia al repositorio. `Train.csv` y `Test.csv` listan cada imagen con su clase y el recuadro (ROI) de la señal. El test oficial se usa solo en la evaluación final.

In [ ]:
from src.data import download_gtsrb, load_tables, split_train_val, GTSRBDataset, get_transforms, CLASS_NAMES, NUM_CLASSES

DATA_ROOT = download_gtsrb()
train_full, test_df = load_tables(DATA_ROOT)
train_df, val_df = split_train_val(train_full, val_size=0.15, seed=SEED)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
print(sorted(p.name for p in DATA_ROOT.iterdir()))
train_full.head()

## 2. Distribución de clases
Se cuenta cuántas imágenes tiene cada clase en train. Un desbalance fuerte puede inflar la accuracy, por eso también se reportan métricas macro (promedio sin ponderar entre clases).

In [ ]:
counts = train_full['ClassId'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(counts.index, counts.values, color='steelblue')
ax.set(xlabel='Clase', ylabel='Imágenes', title='Imágenes por clase (train)'); ax.set_xticks(range(NUM_CLASSES))
plt.tight_layout(); plt.savefig(OUT / 'eda_clases.png', dpi=120); plt.show()
print(f'Mínimo: {counts.min()} ({CLASS_NAMES[counts.idxmin()]}) | Máximo: {counts.max()} ({CLASS_NAMES[counts.idxmax()]}) | Razón: {counts.max()/counts.min():.1f}x')

## 3. Tamaños de imagen
Las imágenes no tienen tamaño fijo. Esto define a qué resolución se redimensionan: 48×48 para la CNN propia y 96×96 para MobileNetV2.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
axs[0].hist(train_full['Width'], bins=50, color='steelblue'); axs[0].set(title='Ancho (px)')
axs[1].hist(train_full['Height'], bins=50, color='indianred'); axs[1].set(title='Alto (px)')
plt.tight_layout(); plt.show()
train_full[['Width', 'Height']].describe().round(1)

## 4. Ejemplo por clase

In [ ]:
raw = GTSRBDataset(train_full, DATA_ROOT)
first_idx = train_full.groupby('ClassId').head(1).index
fig, axs = plt.subplots(5, 9, figsize=(16, 10))
for ax, i in zip(axs.ravel(), first_idx):
    ax.imshow(raw.load_image(i)); ax.set_title(f"{train_full.loc[i, 'ClassId']}: {CLASS_NAMES[train_full.loc[i, 'ClassId']][:18]}", fontsize=7)
for ax in axs.ravel(): ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'eda_ejemplos.png', dpi=120); plt.show()

## 5. Tracks y fuga de datos
Cada archivo se llama `clase_track_frame.png`. Un track son unas 30 fotos consecutivas de la misma señal física conforme el vehículo se acerca. Si la partición se hiciera imagen por imagen, fotos casi iguales quedarían en train y en val. Por eso `split_train_val` separa por track.

In [ ]:
from src.data import add_track_id
tracked = add_track_id(train_full)
one_track = tracked[tracked['track'] == tracked['track'].iloc[0]]
fig, axs = plt.subplots(1, 10, figsize=(16, 2))
for ax, i in zip(axs, one_track.index[::3]):
    ax.imshow(raw.load_image(i)); ax.axis('off')
plt.suptitle(f"Track {one_track['track'].iloc[0]} ({len(one_track)} imágenes)"); plt.show()

shared = set(train_df['track']) & set(val_df['track'])
print('Tracks:', tracked['track'].nunique(), '| compartidos entre train y val:', len(shared))
print('Clases en val:', val_df['ClassId'].nunique())

## 6. Aumento de datos
Se aplican rotaciones pequeñas, traslación, escala y cambios de brillo y contraste. No se usa flip horizontal, porque convierte "girar a la izquierda" en "girar a la derecha".

In [ ]:
from src.data import denormalize
aug = GTSRBDataset(train_df, DATA_ROOT, transform=get_transforms(48, augment=True))
fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for ax in axs.ravel():
    x, y = aug[0]
    ax.imshow(denormalize(x).permute(1, 2, 0)); ax.axis('off')
plt.suptitle(f'Aumentos de una misma imagen · clase {y}: {CLASS_NAMES[y]}'); plt.show()

**Interpretación (completar después de ejecutar):** describa el desbalance observado y qué clases tienen menos ejemplos, el rango de tamaños de imagen y si 48×48 conserva la información de la señal, y por qué la partición por track evita una validación optimista.